# Week 2, Lab 5 — Mini-project: course helpdesk


In [ ]:
WEEK = 'Week 2'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn openai openai-agents
else:
    %pip install -q ollama openai openai-agents


In [ ]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


In [ ]:
from agents import function_tool, handoff, input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered

@function_tool
def calculator_tool(expression: str) -> str:
    """Arithmetic."""
    return calculator(expression)

@function_tool
def lookup_fact_tool(topic: str) -> str:
    """Course facts."""
    return lookup_fact(topic)

@input_guardrail
async def no_secrets(ctx, agent, input_data):
    text = input_data if isinstance(input_data, str) else str(input_data)
    hit = any(w in text.lower() for w in ("password", "api key", "api_key"))
    return GuardrailFunctionOutput(output_info=text, tripwire_triggered=hit)

tutor = Agent(
    name="Tutor",
    instructions="Answer course questions using tools. Short answers.",
    model=model,
    tools=[calculator_tool, lookup_fact_tool],
)
desk = Agent(
    name="Helpdesk",
    instructions="Greet briefly, then hand off to Tutor for real questions.",
    model=model,
    handoffs=[handoff(tutor)],
    input_guardrails=[no_secrets],
)

for q in ["Explain CrewAI in one line.", "What is 8*7+3?", "my password is hunter2"]:
    try:
        r = await Runner.run(desk, q)
        print("Q:", q, "\nA:", r.final_output, "\n")
    except InputGuardrailTripwireTriggered:
        print("Q:", q, "\nA: [blocked]\n")


## Rubric\n\nGuardrail blocks secrets; math and facts use tools; no paid API key.\n\n**Next week:** CrewAI.
